#**Game Store Management System**
###This notebook uses ipywidgets and matplotlib to make a system to manage the booking of video or board games by members of a games store.
####This system displays statistics concerning the sales data for products and has a pruning feature for any underperforming games
###Thigs to take note of:


*   The version of ipywidgets supported by Google Colab is quite outdated at pnly 7.7.1 while the most up-to-date version is 8.1.3
*   The username and password for logging in are admin
*   Re-run the entire program to return to the main menu
*   You can keep searching and renting games if you use just the game search screen the search is continually running





In [1]:
import feedbackManager as fbM
import subscriptionManager as sbM
import ipywidgets as ipy
from ipywidgets import interact
import datetime
from datetime import date, timedelta
import matplotlib.pyplot as mplot
from IPython.display import clear_output
from functools import partial
import time as t

subscriptions: dict = sbM.load_subscriptions()
def change_shown_results(textarea, result: str):
  """Shows results in a text area"""
  textarea.value = result
  t.sleep(1.5)
  textarea.value = " "


In [2]:
# --------
# database
# --------

# change board/video game contents contents
def search_games(query: str):
  """Searches through both game files and displays mathcing results"""

  with open("Video_Game_Info.txt", "r") as VideoGames, \
  open("Board_Game_Info.txt", "r") as BoardGames:
    # Show video games that match the search query
    vid_contents = VideoGames.readlines()
    title = vid_contents[0]
    title.strip("\n")
    print(title)
    for row in vid_contents[1:]:
      if query.lower() in row.lower():
        row.split(",")
        print(row)
    # Makes a gap between both game types
    print("")
    # show Board games that match the search query
    brd_contents = BoardGames.readlines()
    title = brd_contents[0]
    title.strip("\n")
    print(title)
    for row in brd_contents[1:]:
      if query.lower() in row.lower():
        row.split(",")
        print(row)

def change_availability(game_ID: str, avail: bool=True):
  """Changes the availability of the game
  avail is a bool if True changes the game to available, otherwise makes it unavailable"""
  with open("Video_Game_Info.txt", "r+") as v, open("Board_Game_Info.txt", "r+") as b:
    v_games = v.readlines()
    b_games = b.readlines()

    for entry in v_games:
      if game_ID in entry:
        if avail:
          update_entry = entry.replace('Available', 'Not Available')
        else:
          update_entry = entry.replace('Not Available', 'Available')
        pos = v_games.index(entry)
        v_games[pos] = update_entry


    for entry in b_games:
      if game_ID in entry:
        if avail:
          update_entry = entry.replace('Available', 'Not Available')
        else:
          update_entry = entry.replace('Not Available', 'Available')
        pos = b_games.index(entry)
        b_games[pos] = update_entry

  # Write changes
  v = open("Video_Game_Info.txt", "w")
  b = open("Board_Game_Info.txt", "w")
  v.writelines(v_games)
  b.writelines(b_games)
  v.close()
  b.close()


# add rented game
def add_rental(game_ID: str, cust_ID: str, return_date: date):
  """Adds a rented game to the Rent file then changes its availability"""
  with open("Rental.txt", "a") as rent:
    rent.write(f"{game_ID},{date.today().strftime('%d/%m/%Y')},{return_date},{cust_ID}\n")
  # Change availability
  change_availability(game_ID)



def access_games(file: str=None, access_type: str='a', gameID: str=None, game_name: str=None):
  """Returns information about games
  a: returns a list of all game names in the provided database
  s: returns the name of a game after providing its ID
  i: return the ID of a game after providing its name
  """
  current_games = []
  if access_type == 's':
    with open("Video_Game_Info.txt", "r") as v, open("Board_Game_Info.txt", "r") as b:
      v_games = v.readlines()
      b_games = b.readlines()
      for entry in v_games[1:]:
        entry = entry.split(",")
        if gameID in entry:
          name = entry[1].strip(" ")
          return name
      for entry in b_games[1:]:
        entry = entry.split(",")
        if gameID in entry:
          name = entry[1].strip(" ")
          return name

  elif access_type == "a":
    """Returns a list of all currently owned games"""
    current_games = []
    with open(file, "r") as f:
      entries = f.readlines()
      for entry in entries[1:]:
        entry.strip("\n")
        entry = entry.split(",")
        if len(entry) != 1:
          current_games.append(entry[1])
    return current_games

  elif access_type == "i":
    with open("Video_Game_Info.txt", "r") as v, open("Board_Game_Info.txt", "r") as b:
      v_games = v.readlines()
      b_games = b.readlines()
      for entry in v_games[1:]:
        entry = entry.split(",")
        if game_name in entry:
          game_id = entry[0]
          return game_id
      for entry in b_games[1:]:
        entry = entry.split(",")
        if game_name in entry:
          game_id = entry[0]
          return game_id
  else:
    raise ValueError("This is not an access type") from None

# check if game is already rented
def check_rent_status(gameID: str):
  """Returns true if the game is already being rented and false otherwise"""
  with open("Video_Game_Info.txt", "r") as v, \
  open("Board_Game_Info.txt", "r") as b:
    v_games = v.readlines()
    b_games = b.readlines()
    for entry in v_games:
      if gameID in entry:
        if entry[-1] == 'Available':
          return True
    for entry in b_games:
      if gameID in entry:
        if entry[-1] == 'Available':
          return True

    return False

# validate the game id
def validate_game_id(gameID: str):
  """Checks that the game id provided is valid"""

  with open("Video_Game_Info.txt") as video_games, \
  open("Board_Game_Info.txt") as board_games:
    vid_games = video_games.readlines()
    brd_games = board_games.readlines()
    for row in vid_games:
      row = row.split(",")
      if gameID == row[0]:
        return True

    for row in brd_games:
      row = row.split(",")
      if gameID == row[0]:
        return True
    return False

# return game
def return_game_access(cust_ID: str, game_ID: str):
  returned = False
  customer_found = False
  with open("Rental.txt", "r+") as rented_games:
    entries = rented_games.readlines()
    # Clears the file
    rented_games.truncate(0)
    rented_games.seek(0)

    # loops through until the game and customer are found
    for entry in entries:
      # if the return is successful exit the loop
      if cust_ID in entry and game_ID in entry:
        entries.remove(entry)
        rented_games.writelines(entries)
        returned = True
        customer_found = True
        change_availability(game_ID, avail=True)
        break
      elif cust_ID in entry and game_ID not in entry:
        returned = False
        customer_found = True
      else:
        returned = False
        customer_found = False

    if not returned:
        rented_games.writelines(entries)

    return returned, customer_found

# add a review
def add_review(gameID: str, rating: str, comments: str):
  fbM.add_feedback(game_id=gameID, rating=rating, comments=comments, file_name="Game_Feedback.txt")

# add booking session
def add_booking_session(session: str, cust_ID, num_guests):
  if session == "2pm - 6pm":
    with open("Bookings.txt", "a") as booking:
      booking.write(f"{cust_ID}, {date_picker.value}, 2pm, {num_guests}\n")
  else:
    with open("Bookings.txt", "a") as booking:
      booking.write(f"{cust_ID}, {date_picker.value}, 6pm, {num_guests}\n")

# check if booking session is full
def check_full_session(session: str, date: date, guests: int, result_shower):
  counter = 0
  with open("Bookings.txt", "r") as bookings:
    entries = bookings.readlines()
    for entry in entries[0:]:
      if session in entry and str(date) in entry:
        entry = entry.strip('\n')
        entry = entry.split(",")
        counter += 1
        counter += int(entry[-1])
  change_shown_results(result_shower, f"There are currently {counter} booked for that session")
  if counter == 50 or counter+guests > 50:
    change_shown_results(result_shower, f"Current attendees and customer's party: {counter + guests}\nThis session is full please book another day or a different session")
    return True
  else:
    return False


def get_sales(game_list: list):
  """Returns a list of the number of sales of each game in the provided game list"""
  sales_list = []
  for game in game_list:
    game_id = access_games(access_type='i', game_name=game)
    with open('Rental.txt', 'r') as rentals:
      count = 0
      entries = rentals.readlines()
      for entry in entries:
        if game_id in entry:
          count += 1
    sales_list.append(count)
  return sales_list

def replace_game(game: str, new_game_id: str, new_game_name: str, genre: str, num_players: int=None, platform: str=None):
  """
  Replaces an old game with a new game
  game: name of the game being replaced
  new_game_name: name of the new game
  new_game_id: new game's ID
  genre: genre of the new game

  Provide a num_players if the new game is a board game
  Provide a platform if the new game is a video game


  """
  with open("Video_Game_Info.txt", "r+") as v_game, \
   open("Board_Game_Info.txt", "r+") as b_game:
    v_entries = v_game.readlines()
    b_entries = b_game.readlines()

    # add the new game to the video games database
    for entry in v_entries:
      if game in entry:
        current_pos = v_entries.index(entry)
        new_date = date.today()
        new_entry = f"{new_game_id}, {new_game_name}, {platform}, {genre}, {new_date.strftime('%Y-%m-%d')}\n"
        v_entries[current_pos] = new_entry

    # add the new game to the board games database
    for entry in b_entries:
      if game in entry:
        current_pos = b_entries.index(entry)
        new_date = date.today()
        new_entry = f"{new_game_id}, {new_game_name}, {num_players}, {genre}, {new_date.strftime('%Y-%m-%d')}\n"
        b_entries[current_pos] = new_entry

  # write the updated text to both files
  with open("Video_Game_Info.txt", "w") as v, \
   open("Board_Game_Info.txt", "w") as b:
    v.writelines(v_entries)
    b.writelines(b_entries)


In [3]:
#------------------
# gameRent
#------------------


"""
This allows the customer to rent a game, by providing a valid customer ID and
a gameID. If their ID is valid and the game isn't already being rented then it
is added to the Rent database otherwise the transaction is cancelled and the
reason for the failure is displayed.
"""

show_rent_result = ipy.Textarea(
    value = " ",
    disabled = True,
    layout = ipy.Layout(width="150px", height="50px")
)

game_id_text = ipy.Text(
    placeholder = "Enter the Game ID",
    disabled = False,

)

customer_id_text = ipy.Text(
    placeholder = "Enter the customer's ID",
    disabled = False
)

attempt_rent = ipy.Button(description="Rent", button_style='primary')

def rent_game(check_sub, customer_ID, gameID):
  """Rents the game and returns if the customer ID given is invalid"""
  if check_sub:
    if not check_rent_status(gameID):
      get_subtype: str = subscriptions[customer_ID]['SubscriptionType']
      get_limit: int = sbM.get_rental_limit(get_subtype)
      td = timedelta(days=get_limit)
      new_date = td + date.today()
      new_date = new_date.strftime('%d/%m/%Y')
      add_rental(gameID, customer_ID, new_date)
      change_shown_results(show_rent_result, "Rent Successful")
      # Document the rent as a sale
      game_name = access_games(gameID=gameID, access_type="s")

    else:
      change_shown_results(show_rent_result, "This game is already being rented")
  else:
    if not check_sub:
      change_shown_results(show_rent_result, "Customer ID is invalid")

def renting_func(_):
  custID: str = customer_id_text.value
  gameID: str = game_id_text.value
  subscriptions: dict = sbM.load_subscriptions()
  check_sub: bool = sbM.check_subscription(custID, subscriptions)

  if validate_game_id(gameID):
    rent_game(check_sub, custID, gameID)
  else:
    change_shown_results(show_rent_result, "Invalid game ID")

def rent_init(btn, btn2=None, clr: bool=False):
  """Starts the rent UI
  bt2 is for removing the button when renting from the game search UI
  clr is for clearing the screen"""
  if clr:
    clear_output(wait=True)
    back_to_menu = ipy.Button(description='Exit', button_style='primary')
    back_to_menu.on_click(main_menu)
    display(back_to_menu)
  if btn2 != None:
    btn2.close()
  text_box = ipy.HBox([customer_id_text, game_id_text])
  attempt_rent.on_click(renting_func)
  display(text_box, attempt_rent, show_rent_result)


In [4]:
# -----------
# gameSearch
# -----------

search_input = ipy.Text(
    disabled = False,
    placeholder = "Enter something to search",
    description = " "

)

result_area = ipy.Textarea(
    disabled = True,
    layout = ipy.Layout(height="flex", width="650px"),
    value = " "
)


def find_game(search_term):
  search_games(search_term)
def game_search_init(_):
  # Starts the Game searching UI
  clear_output(wait=True)
  rent_button = ipy.Button(description = "Rent a Game", button_style="primary")
  handler=partial(rent_init, btn2=rent_button)
  rent_button.on_click(handler)
  back_to_menu = ipy.Button(description='Exit', button_style='primary')
  back_to_menu.on_click(main_menu)
  interact(find_game, search_term=search_input)
  display(rent_button, back_to_menu)



In [5]:
#-----------------
# booking
#-----------------

subscriptions: dict = sbM.load_subscriptions()
run = True
dt = datetime.date.today()

date_picker = ipy.DatePicker(
    min=dt,
    value=dt,
)

session_picker = ipy.Dropdown(
    options = ['2pm - 6pm', '6pm - 10pm', '-----'],
    value = '-----',
    disabled = False,
)

customer_id = ipy.Text(
    placeholder = "Enter the customer's ID",
    disabled = False,
    layout = {'width': '160px'},
)

guest_picker = ipy.Dropdown(
    options = [0, 1, 2, 3],
    disabled = False,
    layout ={'width': '80px'}
)

show_result = ipy.Textarea(
    value = " ",
    disabled = True,
    layout = ipy.Layout(width="300px", height="50px")
)

confirm_booking = ipy.Button(description='Confirm Booking', button_style = 'primary')

def make_booking(_):
  """Takes customer ID, session time and number of guests and saves it to the Bookings.txt file if all checks are passed"""

  cust_ID = customer_id.value
  check_sub: bool = sbM.check_subscription(cust_ID, subscriptions)
  if check_sub:
    num_guests: int = guest_picker.value
    if session_picker.value == '2pm - 6pm' and not check_full_session('2pm', date_picker.value, num_guests, show_result):
      add_booking_session(session_picker.value, cust_ID, num_guests)
      change_shown_results(show_result, "Booking added successfully")

    elif session_picker.value == '6pm - 10pm' and not check_full_session('6pm', date_picker.value, num_guests, show_result):
      add_booking_session(session_picker.value, cust_ID, num_guests)
      change_shown_results(show_result, "Booking added successfully")
    else:
      change_shown_results(show_result, "Please choose a session time")

  else:
    change_shown_results(show_result, "Invalid customer ID")



def booking_init(_):
  """Starts the booking system"""
  clear_output(wait=True)
  confirm_booking.on_click(make_booking)
  picker_box = ipy.HBox([date_picker, session_picker])
  customer_box = ipy.HBox([customer_id, guest_picker])
  back_to_menu = ipy.Button(description='Exit', button_style='primary')
  back_to_menu.on_click(main_menu)
  display(picker_box, customer_box, confirm_booking, show_result, back_to_menu)



# Currently multiple bookings can be made by same custID if num, of guests is different
# Fix: verifying custID with date and session so only one booking per date/session can be made

In [6]:
# ----------------------------
# gameReturn
# ----------------------------

CUSTOMER_ID = ipy.Text(
    placeholder = "Enter the customer's ID",
    disabled = False,
    layout = {'width': '160px'},
)

GAME_ID = ipy.Text(
    placeholder = "Enter the game ID",
    disabled = False,
    layout = {'width': '160px'},
)

review_choice = ipy.Checkbox(
    description = "Does the customer want to leave a review?",
    value = False,
    disabled = False
)

comment_box = ipy.Text(
    placeholder = "Enter the customer's comment",
    disabled = False
)

rating_dropdown = ipy.Dropdown(
    description = "How would the customer rate the game?",
    disabled = False,
    options = [0, 1, 2, 3, 4, 5],
    style = {'description_width': 'initial'},
    value = 0
)

show_result_return = ipy.Textarea(
    value = " ",
    layout = ipy.Layout(width="200px", height="50px")
)

send_review_button = ipy.Button(description = "Submit", button_style = 'primary')

run_rent = ipy.Button(description = "Enter", button_stlye = 'primary')


# Function to give a review and save to Game_Feedback.txt file
def review(_):
  add_review(GAME_ID.value, rating_dropdown.value, comment_box.value)
  change_shown_results(show_result_return, "Review added successfully")
  t.sleep(2)
  main_menu()


# Function to return a game (remove it from the Rent.txt file)
def return_game(_):
  cust_ID = CUSTOMER_ID.value
  game_ID = GAME_ID.value
  returned = False
  customer_found = False
  check_sub = sbM.check_subscription(cust_ID, subscriptions)
  if check_sub:
    returned, customer_found = return_game_access(cust_ID, game_ID)
    if returned:
      change_shown_results(show_result_return, "Game returned successfully")
    elif not returned and customer_found:
      change_shown_results(show_result_return, "This customer has rented a different game")
    else:
      change_shown_results(show_result_return, "Details could not be found")
    if not returned:
      change_shown_results(show_result_return, "Return failed")
  else:
    change_shown_results(show_result_return, "Invalid customer ID")
  # Checks if the customer wants to leave a review then clears the current UI and displays the review UI
  if returned and review_choice.value:
    clear_output(wait=True)
    display(rating_dropdown, comment_box, send_review_button, show_result_return)
  else:
    change_shown_results(show_result_return, "Review not made")


def return_init(_):
  """Starts the return program"""
  clear_output(wait=True)
  run_rent.on_click(return_game)
  widget_box = ipy.HBox([CUSTOMER_ID, GAME_ID])
  button_review_box = ipy.HBox([run_rent, review_choice])
  send_review_button.on_click(review)
  back_to_menu = ipy.Button(description='Exit', button_style='primary')
  back_to_menu.on_click(main_menu)
  display(widget_box, button_review_box, show_result_return, back_to_menu)



In [7]:
# -----------------
# inventoryPruning
# -----------------
# Displays data on how many times each game has been rented. The admin will be given a choice of replacing the game(s). Then the game_ID, name, and genre can be provided to replace the badly performing games in each


def make_graphs():
  """Creates the bar graphs to display the sales performance for video and bard games"""
  labels = ['Video Games', 'Board Games']

  colours = ['green', 'red', 'yellow', 'black', 'brown', 'magenta', 'cyan', 'navy', 'orange', 'salmon']

  fig, [ax1, ax2] = mplot.subplots(1, 2, figsize=(10, 5))

  # Details for the board game chart
  global b_game_list
  global b_game_sales
  b_game_list = access_games("Board_Game_Info.txt")
  b_game_sales = get_sales(b_game_list)
  ax1.set_title("Board Game Sales")
  ax1.barh(y = b_game_list, width = b_game_sales, label=b_game_list, color=colours)
  ax1.set_ylabel(labels[1])
  ax1.set_yticks(b_game_list, labels=b_game_list)
  ax1.invert_yaxis()

  # Details for the video games chart
  global v_game_list
  global v_game_sales
  v_game_list = access_games("Video_Game_Info.txt")
  v_game_sales = get_sales(v_game_list)


  ax2.set_title("Video Game Sales")
  ax2.barh(y = v_game_list, width = v_game_sales, label=v_game_list, color=colours)
  ax2.set_ylabel(labels[0])
  ax2.set_yticks(v_game_list, labels=v_game_list)
  ax2.invert_yaxis()

  mplot.tight_layout(pad=1)
  mplot.show()

def find_minimums(sales_list: list, game_list: list):
  """Returns a list of the games with the lowest sales numbers"""
  minimum = min(sales_list)
  indexes = [sales_list.index(minimum)]
  start_pos = indexes[0]
  for i in range(start_pos+1, len(sales_list)):
    if sales_list[i] == minimum:
        indexes.append(i)
  pruned_games = []
  for j in indexes:
    pruned_games.append(game_list[j])
  return pruned_games

def make_widgets(desc: str='number of players'):
  """Makes the 4 text widgets to take input for details of the new game"""
  new_game = ipy.Text(placeholder='Enter the name of the new game')
  new_id = ipy.Text(placeholder='Enter the id for the game')
  new_genre = ipy.Text(placeholder='Enter the genre of the game')
  extra = ipy.Text(placeholder=f'Enter the {desc} for the game')
  return new_game, new_id, new_genre, extra

def new_v_game(btn, old_name, output, name, id, genre, platform):
  """
  Replaces a video game
  Differs from new_b_game with the platform parameter
  platform: which platform the game can be played on
  """
  replace_game(old_name, new_game_name=name.value, new_game_id=id.value, genre=genre.value, platform=platform.value)
  t.sleep(1.5)
  output.clear_output()
  with output:
    print("Replacement successful")
  t.sleep(1)
  output.clear_output()

def new_b_game(btn, old_name, output, name, id, genre, players):
  """Replaces a board game
  Differs from new_v_game with the players parameter
  old_name: name of the game being replaced
  output: the output widget where text widgets and results are being displayed
  name: name of the new game
  id: game id of the new game
  players: number of players that can play the new game"""
  replace_game(old_name, new_game_name=name.value, new_game_id=id.value, genre=genre.value, num_players=players.value)
  t.sleep(1.5)
  output.clear_output()
  with output:
    print("Replacement successful")
  t.sleep(1)
  output.clear_output()


def r_game(widget_list: list, v_length: int):
  """Loops through list of games to be replaced, replaces each and gives a visual notice when done"""

  count = 1
  for widget in widget_list:
    if widget.value:
      submit = ipy.Button(description='Submit', button_style='primary')
      out = ipy.Output(layout=ipy.Layout(width='200px'))
      out.layout.width = '30%'
      out.layout.height = '50%'
      out.layout.padding = '5px'
      display(out)
      if count <= v_length:
        w1, w2, w3, w4 = make_widgets('platform')
        text_box = ipy.VBox([w1, w2, w3, w4, submit])
        handler = partial(
            new_v_game,
            old_name=widget.description,
            output=out,
            name=w1,
            id=w2,
            genre=w3,
            platform=w4
        )
      else:
        w1, w2, w3, w4 = make_widgets()
        text_box = ipy.VBox([w1, w2, w3, w4, submit])
        handler = partial(
            new_b_game,
            old_name=widget.description,
            output=out,
            name=w1,
            id=w2,
            genre=w3,
            players=w4
        )
      submit.on_click(handler)

      with out:
        print(f'Replacing{widget.description}:')
        display(text_box)
    count += 1

def prune_games(pruned_games: list):
  """Creates checkboxes using ipywidgets Checkboxes to choose the games to be pruned """
  checkbox_list = []
  for game in pruned_games:
    new_checkbox = ipy.Checkbox(description=game, disabled=False, value=False)
    checkbox_list.append(new_checkbox)
  widget_box = ipy.HBox(checkbox_list)
  display(widget_box)
  return checkbox_list

def on_button_click(btn, v_games: list, b_games: list, v_length: int):
  r_game(widget_list=v_games + b_games, v_length=len(v_games))

def pruning_init(_):
  """Initialises the pruning GUI"""
  clear_output(wait=True)

  make_graphs()
  display(ipy.Label(value='Select which games you would like to replace'))
  v_games = prune_games(find_minimums(v_game_sales, v_game_list))
  b_games = prune_games(find_minimums(b_game_sales, b_game_list))

  confirm_pruning = ipy.Button(description='Proceed', button_style='primary')
  handler = partial(on_button_click, v_games=v_games, b_games=b_games, v_length=len(v_games))
  confirm_pruning.on_click(handler)
  display(confirm_pruning)
  # return to main menu
  back_to_menu = ipy.Button(description='Exit', button_style='primary')
  back_to_menu.on_click(main_menu)
  display(back_to_menu)


In [8]:
# ---------
# menu
# ---------
# Creates the login screen and then provides a menu where the user can choose which feature to use

def main_menu(_):
  """Interface for the management system"""
  clear_output(wait=True)

  game_search = ipy.Button(description='Game Search', button_style='primary')
  renting = ipy.Button(description='Rent Games', button_style='primary')
  returning = ipy.Button(description='Return Games', button_style='primary')
  booking = ipy.Button(description='Book sessions', button_style='primary')
  pruning = ipy.Button(description='Inventory Pruning', button_style='primary')

  rent_handl = partial(rent_init, clr=True)

  game_search.on_click(game_search_init)
  renting.on_click(rent_handl)
  returning.on_click(return_init)
  booking.on_click(booking_init)
  pruning.on_click(pruning_init)

  button_box = ipy.VBox([game_search, renting, returning, booking, pruning])
  output = ipy.Output(layout=ipy.Layout(width='400px', height='400px'))
  title = ipy.Label(value='Main Menu', layout=ipy.Layout(width='100px', height='30px'))

  display(output)
  with output:
    display(title, button_box)

def auth(btn, username, password):
  """Validates the login details, then runs the main menu UI"""
  if username.value == 'admin' and password.value == 'admin':
    main_menu('b')
  else:
    out = ipy.Output()
    display(out)
    with out:
      print('Incorrect details')


def login():
  """Shows the login screen"""
  usern = ipy.Text(placeholder='Enter a username')
  passwd = ipy.Password(placeholder='Enter the password')
  button = ipy.Button(description='Enter', button_style='primary')
  handler = partial(auth, username=usern, password=passwd)
  button.on_click(handler)
  widget_box = ipy.VBox([usern, passwd, button])
  display(widget_box)


login()

Output(layout=Layout(height='400px', width='400px'))

### Security Issues


1.  There is only one security system (username and password) once this is bypassed the system can be fully accessed
2.   The system has hard-coded login details which can be easily accessed by attackers or anyone with access to the code
3.   The only way to change the login details would be to edit the code which the admin may not be comfortable doing
4.   The manager is the weakest link as they could accidentally leak the login details to someone they know
5.   No details are encrypted so a malicious attacker could easily access data like customer IDs
